# data-collection_001

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (1).ipynb`

| Field | Value |
|---|---|
| Section | `data_collection` |
| Market | `us_equities` |
| Trading style | `unknown` |
| Timeframe | `1m` |
| Code cells | 19 |
| Detected functions | — |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas_datareader as pdr
import yfinance as yf
import datetime

In [ ]:
df = yf.download(tickers=['^NSEI'], start = '2024-10-14', end = '2024-10-19', interval="1m")
df.head()

In [ ]:
df = df.drop(['Open', 'High', 'Low', 'Adj Close', 'Volume'], axis = 1)
df

In [ ]:
plt.figure(figsize = (12, 6))
plt.plot(df.Close)

In [ ]:
df['5-min'] = df['Close'].rolling(5).mean()
df['3-min'] = df['Close'].rolling(3).mean()
df

In [ ]:
df['signal'] = np.where(df['3-min'] > df['5-min'], 1, 0)
df['signal'] = np.where(df['3-min'] < df['5-min'], -1, df['signal'])
df.dropna(inplace=True)
df.head()

In [ ]:
df['return'] = np.log(df['Close']).diff()
df['signal-return'] = df['return'] * df['signal']
df.tail()

In [ ]:
df['entry'] = df.signal.diff()
df

In [ ]:
plt.figure(figsize = (13, 7))
plt.plot(df.Close)
plt.plot(ma5, 'r')
plt.plot(ma3, 'y')
plt.plot(df['3-min'][df.entry == 2], '^', color = 'g', markersize = 7)
plt.plot(df['5-min'][df.entry == -2], '^', color = 'r', markersize = 7)

In [ ]:
df['buy'] = (df['3-min'][df.entry == 2]).astype(int)
df['sell'] = (df['3-min'][df.entry == -2]).astype(int)
df[1:50]

In [ ]:
df.buy.unique()

In [ ]:
df['trade'] = df['buy'].fillna(df['sell']).combine_first(df['sell'])
df

In [ ]:
df.trade.count()

In [ ]:
# Drop NaN values but keep the index for later
valid_numbers = df['trade'].dropna()

# Calculate the difference between consecutive numbers
differences = valid_numbers.diff()

# Create a new column with NaN where the original column had NaN, and differences where numbers existed
df['difference'] = differences.reindex(df.index)


In [ ]:
df

In [ ]:
df.difference.unique()

In [ ]:
df.difference.sum()/df.Close.mean() * 100

In [ ]:
df.Close.mean()